# TRPO ve PPO Teorisi

TRPO ve PPO, policy gradient update'lerinin politikayı tek adımda fazla değiştirmesini engellemek için geliştirilmiştir. PPO, TRPO'nun pratik ve kolay uygulanabilir bir yaklaşımıdır.

## 1. Neden Trust Region?

PPO on-policy actor-critic ailesindedir. Toplanan rollout, `old policy` tarafından üretilir; update sırasında yeni politika bu eski politikadan kontrollü biçimde uzaklaştırılır. Bu yüzden PPO kısa süreli veri tekrarına izin verir, ama DQN tarzı uzun ömürlü replay buffer algoritması değildir.

Vanilla policy gradient tek update'te politikayı çok değiştirebilir. Yeni politika, rollout'u üreten eski politikadan çok uzaklaşırsa advantage tahminleri güvenilmez hale gelir.

Bu nedenle hedef şudur:

- İyi aksiyonların olasılığını artır.
- Kötü aksiyonların olasılığını azalt.
- Ama yeni politika eski politikadan çok uzaklaşmasın.

## 2. Importance Sampling Ratio

Rollout eski politika ile toplanır:

$$\pi_{old}(a\mid s)$$

Yeni politika için objective şu oranla yazılır:

$$r_t(\theta)=\frac{\pi_\theta(a_t\mid s_t)}{\pi_{old}(a_t\mid s_t)}=\exp(\log\pi_\theta-\log\pi_{old})$$

Surrogate objective:

$$L(\theta)=\mathbb{E}[r_t(\theta)\hat{A}_t]$$

## 3. TRPO

TRPO şu kısıtlı optimizasyonu çözer:

$$\max_\theta\mathbb{E}[r_t(\theta)\hat{A}_t]$$

$$\text{subject to }\mathbb{E}[KL(\pi_{old}(\cdot\mid s),\pi_\theta(\cdot\mid s))]\leq\delta$$

Yani objective iyileştirilir, fakat KL uzaklığı belirli bir sınırı aşamaz. TRPO teorik olarak güçlüdür ama conjugate gradient, Fisher-vector product ve line search gibi uygulama karmaşıklıkları içerir.

## 4. PPO Clipped Objective

PPO, hard KL constraint yerine ratio clipping kullanır:

$$L^{CLIP}=\mathbb{E}\left[\min(r_t\hat{A}_t,clip(r_t,1-\epsilon,1+\epsilon)\hat{A}_t)\right]$$

Eğer $\hat{A}_t>0$ ise ratio'nun çok büyümesi daha fazla ödüllendirilmez. Eğer $\hat{A}_t<0$ ise ratio'nun çok küçülmesi daha fazla ödüllendirilmez. Bu, politikayı aşırı güncelleme riskini azaltır.

### 4.1. Clipping Sezgisi

| Advantage işareti | Ratio ne demek? | Clipping etkisi |
| --- | --- | --- |
| $\hat{A}_t>0$ | Aksiyon beklenenden iyi | Yeni politika bu aksiyonu daha olası yapabilir, ama $1+\epsilon$ üstü ekstra kazanç getirmez |
| $\hat{A}_t<0$ | Aksiyon beklenenden kötü | Yeni politika bu aksiyonu daha az olası yapabilir, ama $1-\epsilon$ altı ekstra kazanç getirmez |
| $\hat{A}_t\approx0$ | Aksiyon ortalama civarı | Ratio değişse bile policy gradient sinyali zayıftır |

Bu tablo PPO'nun temel güvenlik mantığını özetler: iyi/kötü sinyali takip edilir, fakat eski politikadan çok uzaklaşan adımların objective kazancı kırpılır.

In [ ]:
import torch

def ppo_clip_terms(new_logp, old_logp, advantage, eps=0.2):
    ratio = torch.exp(new_logp - old_logp)
    unclipped = ratio * advantage
    clipped = torch.clamp(ratio, 1.0 - eps, 1.0 + eps) * advantage
    objective = torch.min(unclipped, clipped)
    return ratio, unclipped, clipped, objective

new_logp = torch.tensor([-0.6, -1.5, -0.9])
old_logp = torch.tensor([-1.0, -1.0, -1.0])
advantage = torch.tensor([1.0, -1.0, 0.5])
ppo_clip_terms(new_logp, old_logp, advantage)

## 4.2. PPO Actor-Critic Neural Network

PPO ağının mimarisi Actor-Critic'e benzer; fark update formülündedir. Rollout sırasında `old_log_prob` ve `old_value` saklanır. Update sırasında aynı aksiyonlar yeni ağ ile tekrar değerlendirilir ve ratio clipping uygulanır.

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.distributions import Categorical
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Bu hücre için PyTorch gerekli. Jupyter'da orbit_wars_env kernelini seçin.") from exc

class PPOActorCritic(nn.Module):
    """PPO: ortak trunk + actor logits + critic value."""
    def __init__(self, obs_dim=12, hidden_dim=64, n_actions=5):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
        )
        self.actor = nn.Linear(hidden_dim, n_actions)
        self.critic = nn.Linear(hidden_dim, 1)

    def evaluate_actions(self, obs, actions):
        h = self.trunk(obs)
        dist = Categorical(logits=self.actor(h))
        value = self.critic(h).squeeze(-1)
        return dist.log_prob(actions), dist.entropy(), value

policy = PPOActorCritic()
obs = torch.randn(8, 12)
actions = torch.tensor([0, 1, 2, 3, 4, 0, 1, 2])
advantages = torch.randn(8)
returns = torch.randn(8)

# Rollout'ta saklanmış eski değerler gibi düşün.
with torch.no_grad():
    old_logp, _, old_value = policy.evaluate_actions(obs, actions)
    old_logp = old_logp + 0.05 * torch.randn_like(old_logp)  # eski politika farkını simüle et

new_logp, entropy, value = policy.evaluate_actions(obs, actions)
ratio = torch.exp(new_logp - old_logp)
unclipped = ratio * advantages
clipped = torch.clamp(ratio, 0.8, 1.2) * advantages
policy_loss = -torch.min(unclipped, clipped).mean()
value_loss = 0.5 * F.mse_loss(value, returns)
entropy_bonus = entropy.mean()
total_loss = policy_loss + 0.5 * value_loss - 0.01 * entropy_bonus

print("ratio min/max:", float(ratio.min().detach()), float(ratio.max().detach()))
print("policy_loss:", float(policy_loss.detach()))
print("value_loss:", float(value_loss.detach()))
print("total_loss:", float(total_loss.detach()))

## 4.3. PPO Update Algoritması

PPO update'i rollout sırasında saklanan `old_logp` değerlerini kullanır. Aynı batch üzerinde birkaç epoch çalışılabilir; burada tek minibatch/tek epoch gösteriliyor.

In [ ]:
def ppo_update_step(policy, optimizer, obs, actions, old_logp, returns, advantages, clip_range=0.2, value_coef=0.5, entropy_coef=0.01):
    new_logp, entropy, values = policy.evaluate_actions(obs, actions)
    ratio = torch.exp(new_logp - old_logp)

    unclipped = ratio * advantages
    clipped = torch.clamp(ratio, 1.0 - clip_range, 1.0 + clip_range) * advantages
    policy_loss = -torch.min(unclipped, clipped).mean()
    value_loss = 0.5 * (values - returns).pow(2).mean()
    entropy = entropy.mean()
    loss = policy_loss + value_coef * value_loss - entropy_coef * entropy

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=0.5)
    optimizer.step()

    with torch.no_grad():
        approx_kl = 0.5 * (old_logp - new_logp).pow(2).mean()
        clip_fraction = ((ratio - 1.0).abs() > clip_range).float().mean()

    return {
        "loss": float(loss.detach()),
        "policy_loss": float(policy_loss.detach()),
        "value_loss": float(value_loss.detach()),
        "entropy": float(entropy.detach()),
        "approx_kl": float(approx_kl.detach()),
        "clip_fraction": float(clip_fraction.detach()),
    }

policy = PPOActorCritic(obs_dim=12, n_actions=5)
optimizer = torch.optim.Adam(policy.parameters(), lr=1e-3)
obs = torch.randn(16, 12)
with torch.no_grad():
    logits = policy.trunk(obs)
    dist = Categorical(logits=policy.actor(logits))
    actions = dist.sample()
    old_logp = dist.log_prob(actions)
returns = torch.randn(16)
advantages = torch.randn(16)
ppo_update_step(policy, optimizer, obs, actions, old_logp, returns, advantages)

## 5. Value Clipping

PPO bazı uygulamalarda value function update'ini de sınırlar:

$$V_{clip}=V_{old}+clip(V_\theta-V_{old},-\epsilon_v,\epsilon_v)$$

Value loss:

$$L_V=\frac{1}{2}\max((V_\theta-R)^2,(V_{clip}-R)^2)$$

Orbit Wars `rl/ppo.py` içinde `value_clip_range` ile bu seçenek desteklenir.

## 6. KL ve Clip Fraction Takibi

PPO'da iki sağlık metriği önemlidir:

- `approx_kl`: eski ve yeni politika arasındaki yaklaşık uzaklık
- `clip_fraction`: örneklerin ne kadarında ratio clipping devreye girmiş

`approx_kl` çok yüksekse politika fazla hızlı değişiyor olabilir. `clip_fraction` sürekli yüksekse learning rate fazla, batch küçük veya advantage ölçeği problemli olabilir.

## 7. Orbit Wars PPO Uygulaması

`rl/ppo.py::ppo_update` şu akışı uygular:

1. Rollout batch'inden eski log-prob, advantage ve return okunur.
2. Güncel politika aynı aksiyonlar için yeni log-prob üretir.
3. Ratio hesaplanır: `exp(new_logp - old_logp)`.
4. Clipped surrogate policy loss hesaplanır.
5. Value loss ve entropy bonus eklenir.
6. Gradient clipping sonrası optimizer adımı atılır.

Aksiyon dağılımı `src`, `tgt`, `ship_bin` branch'lerinden oluşur; PPO ratio joint log-prob üzerinden hesaplanır.

## 8. TRPO ve PPO Karşılaştırması

| Özellik | TRPO | PPO |
| --- | --- | --- |
| Kısıt | Açık KL constraint | Ratio clipping |
| Uygulama | Daha karmaşık | Daha basit |
| Optimizer | İkinci mertebe yaklaşımlar | Adam/SGD ile kolay |
| Stabilite | Güçlü teorik motivasyon | Pratikte çok güçlü |
| Popülerlik | Daha az yaygın | Modern varsayılanlardan biri |